In [ ]:
import tensorflow as tf
import numpy as np
import pickle
import keras
import sys
sys.path.append("D://CoreOutline/transformer")
from text_classification_model import TransformerBlock, TokenAndPositionEmbedding, Hestia
import pandas as pd
from itertools import chain
from sklearn.model_selection import train_test_split

In [ ]:
hestia = Hestia(20000, 518)

In [ ]:
#model = hestia.load_model_weights("D://CoreOutline/transformer/ad-hoc-notebooks/model.weights.h5")

In [ ]:
#model.summary()

In [ ]:
#model.weights

In [ ]:
hestia.pop_layer()

In [ ]:
#hestia.freeze_layers()

In [ ]:
hestia.load_tokenizer("D://CoreOutline/transformer/tokenizer")

In [ ]:
df = pd.read_csv("D://CoreOutline/data/bitext_customer_support.csv")

In [ ]:
df['response_no_punctuation'] = df['response'].apply(hestia.remove_punctuation)

In [ ]:
df['response_lowercase'] = df['response_no_punctuation'].str.lower()

In [ ]:
df['response_tokenized'] = df['response_lowercase'].apply(hestia.word_tokenize_text)

In [ ]:
df['response_no_stopwords'] = df['response_tokenized'].apply(hestia.remove_stopwords)

In [ ]:
df['response_encoded'] = [ list(chain(*hestia.tokenizer.texts_to_sequences(i))) for i in df['response_no_stopwords'] ]

In [ ]:
df['response_encoded']

In [ ]:
pad_token = 0  # The token to use for padding
max_length = max(len(seq) for seq in df['response_encoded']) 

In [ ]:
max_length

In [ ]:
df['response_padded'] = [ hestia.padding(i,518)  for i in df['response_encoded']]

In [ ]:
df[['response_padded']]

In [ ]:
hestia.labelencode(df['intent'])

In [ ]:
df['category_le'] = hestia.labelencoder.fit_transform(df['intent'])

In [ ]:
df['category_le']

In [ ]:
X = np.array(pd.DataFrame([ i for i in df['response_padded']]))
y=  np.array(df['category_le'])

In [ ]:
X_train, X_test_val, y_train, y_test_val = train_test_split(X,y)
X_test, X_val, y_test, y_val = train_test_split(X_test_val,y_test_val)

In [ ]:
classes = len(df['category_le'].unique())

In [ ]:
hestia.append_layer(classes)

In [ ]:
hestia.set_callbacks()

In [ ]:
hestia.train( X_train, y_train, X_val, y_val)